# Chat Models 基础用法

对标文档：[LangChain > Core Components > Models / Messages](https://langchain-doc.cn/v1/python/langchain/overview.html)

In [2]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_learning.llm import get_llm

llm = get_llm()

**术语：**
- **LLM** = Large Language Model（大语言模型），如 ChatGPT、DeepSeek
- **get_llm()** = 项目封装的函数，自动读取 API Key 并创建 LLM 实例
- **HumanMessage / SystemMessage** = 消息类型，分别表示"用户消息"和"系统设定"

`get_llm()` 从工具模块中获取 LLM 实例，默认使用 DeepSeek 模型。可以通过 `model=` 参数切换其他模型。

### 1. 简单调用

In [3]:
response = llm.invoke("用一句话解释什么是 LangChain")
print(response.content)

LangChain是一个开源框架，用于简化基于大语言模型（LLM）的应用开发，通过提供模块化组件（如链、代理、记忆）来连接模型、数据源和外部工具。


**术语：**
- **invoke** = "调用"，让 LLM 处理输入并返回结果
- **.content** = LLM 返回的消息对象中的文本内容

`llm.invoke()` 是最基本的调用方式，传入字符串直接获取模型的完整回复。类似给 AI 发了一条消息等它回复。

### 2. 多消息对话

使用 SystemMessage 设定角色，HumanMessage 传入用户输入。

In [4]:
messages = [
    SystemMessage("你是一个 Python 导师，回答简洁有代码示例"),
    HumanMessage("LCEL 中的 | 操作符是什么意思？"),
]
response = llm.invoke(messages)
print(response.content)

`|` 操作符在 LCEL (LangChain Expression Language) 中用于**链式组合**不同的组件，将前一个组件的输出传递给下一个组件作为输入。

```python
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# 创建链：提示词 → 模型 → 输出解析器
chain = prompt | model | StrOutputParser()

# 等价于：
# chain = prompt.invoke(input) → model.invoke(result) → StrOutputParser().invoke(result)
```

简单说就是**数据流管道**，类似 Unix 的管道操作符。


**术语：**
- **SystemMessage** = 系统消息，用来设定 AI 的角色和行为（只对本次对话生效）
- **HumanMessage** = 用户消息，模拟人的提问

通过 `SystemMessage` 设定角色行为，用 `HumanMessage` 传入用户问题，构建多轮对话。模型会根据 system prompt 的角色定位来回答问题。

### 3. 流式输出

In [5]:
for chunk in llm.stream("从 1 数到 5，每个数字一行"):
    print(chunk.content, end="", flush=True)
print()

好的，我们按你的要求，从 1 数到 5，每个数字单独一行：

1  
2  
3  
4  
5


**术语：**
- **stream** = "流式输出"，一个字一个字地返回结果，而不是等全部生成完
- **chunk** = 数据块，stream 每次返回的一小段内容

`llm.stream()` 以流式方式逐块返回输出，适合实时展示生成内容——就像打字机效果。`end=""` 让每个 chunk 连续打印不换行。

### 4. 多供应商切换

`ChatOpenAI` 支持任意兼容 OpenAI API 的服务，只需要改 `base_url`。

In [6]:
# 切换 DeepSeek R1（推理模型）
# r1_llm = get_llm(model="deepseek-reasoner")
# print(r1_llm.invoke("9.9 和 9.11 哪个大？").content)

**术语：**
- **DeepSeek Reasoner** = DeepSeek 的推理模型（R1），擅长逻辑推理
- **model 参数** = 指定使用哪个模型

切换方式只需传入 `model="deepseek-reasoner"`。这里被注释掉了，取消注释即可运行。